<p align="center">
  <a href="https://www.plus2net.com/python/pandas-dataframe-fillna.php">
    <img src="https://www.plus2net.com/images/top2.jpg" alt="plus2net Python Tutorials" width="300">
  </a>
</p>

# Pandas fillna(): Handle Missing Values

**Pandas Tutorial Series — Data Cleaning**

Full tutorial:  
https://www.plus2net.com/python/pandas-dataframe-fillna.php

Learn how to fill missing values without losing the meaning of your data. All datasets here are synthetic teaching examples, not customer records.

## How to Use This Notebook in Google Colab

- Run the setup cell first, then work from top to bottom.
- Predict each result before running its code cell.
- Change the values and run examples again.
- Examples create fresh data so one cleaning choice does not change another example.
- Use the practice area before checking the suggested solution.
- In Colab, save a copy to your Drive to keep your edits. Download the notebook through the File menu if needed.
- No Drive mounting or database connection is required. The sales example downloads its CSV automatically from Plus2Net; an internet connection is needed for that cell.

The setup cell displays your installed versions. Exact table spacing and dtype labels can differ by version.

## What You Will Learn

- Identify missing values with `isna()`.
- Fill numeric and text columns with appropriate values.
- Apply different rules to different columns.
- Compare zero filling and median filling.
- Use `ffill()` and `bfill()` for ordered data.
- Limit replacements and align replacement values by index.
- Understand `inplace=True` and preserve original data.
- Clean a sample sales CSV and verify the results.
- Explain when a missing value should remain missing.

## Setup

Import Pandas and check the environment. If the import fails in a custom runtime, install Pandas before proceeding. These examples use `ffill()` and `bfill()` rather than the older `fillna(method=...)` form.

In [ ]:
import pandas as pd
import platform

print("Python:", platform.python_version())
print("Pandas:", pd.__version__)

## 1. Identify Missing Values

Both `None` and numeric missing values are detected by `isna()`. An empty string is not automatically a missing value.

In [ ]:
df = pd.DataFrame({
    "product": ["Notebook", None, "Pen", ""],
    "quantity": [2.0, None, 4.0, 1.0],
})
print(df.to_string(index=False))
print("\nMissing values per column:")
print(df.isna().sum().to_string())

**Expected output**

```text
 product  quantity
Notebook       2.0
     NaN       NaN
     Pen       4.0
               1.0

Missing values per column:
product     1
quantity    1
```

The empty product string is not counted. If blank strings mean missing in your dataset, normalise them before filling.

### Try It Yourself
Replace the empty string with `None` and compare the missing-value count.

## 2. Fill a Text Column

Use an explicit string dtype and fill an unknown category with a meaningful label.

In [ ]:
products = pd.Series(["Notebook", None, "Pen"], dtype="string")
filled = products.fillna("Unknown")
print(filled.tolist())
print("Original missing count:", int(products.isna().sum()))

**Expected output**

```text
['Notebook', 'Unknown', 'Pen']
Original missing count: 1
```

The original Series is unchanged. A label such as "Unknown" preserves the distinction between a known and unknown product.

## 3. Fill Numeric Values When Zero Is Meaningful

In this example, a missing discount means no discount under the stated business rule. That does not imply every missing numeric value should become zero.

In [ ]:
discount = pd.Series([5.0, None, 0.0, 10.0])
print(discount.fillna(0).tolist())

**Expected output**

```text
[5.0, 0.0, 0.0, 10.0]
```

A missing quantity or price could mean unknown information. Do not automatically apply the discount rule to those columns.

## 4. Apply Different Rules to Different Columns

A dictionary maps column names to their replacement values. Unspecified columns keep their missing values.

In [ ]:
df = pd.DataFrame({
    "category": pd.Series(["Stationery", None, "Stationery"], dtype="string"),
    "discount": [5.0, None, 0.0],
    "unit_price": [20.0, None, 10.0],
})
cleaned = df.fillna({"category": "Unknown", "discount": 0})
print(cleaned.to_string(index=False))
assert cleaned["unit_price"].isna().sum() == 1

**Expected output**

```text
  category  discount  unit_price
Stationery       5.0        20.0
   Unknown       0.0         NaN
Stationery       0.0        10.0
```

The unknown price remains missing. Filling numeric columns with text such as "D" can change their dtype or fail for typed columns; use column-appropriate replacements.

## 5. Compare Zero Filling with Median Filling

The median is one possible estimate, not a recovered observation. Compare how two choices change the average.

In [ ]:
scores = pd.Series([10.0, 20.0, None, 30.0])
zero_filled = scores.fillna(0)
median_filled = scores.fillna(scores.median())
print("Observed-value mean:", scores.mean())
print("Mean after zero fill:", zero_filled.mean())
print("Mean after median fill:", median_filled.mean())
print("Median-filled values:", median_filled.tolist())

**Expected output**

```text
Observed-value mean: 20.0
Mean after zero fill: 15.0
Mean after median fill: 20.0
Median-filled values: [10.0, 20.0, 20.0, 30.0]
```

Pandas excludes missing values from these mean and median calculations. Median filling can distort the distribution. An all-missing column has no observed median, so this strategy cannot fill it.

### Try It Yourself
Change `30.0` to `300.0` and compare the results. Which assumptions justify filling?

## 6. Forward Fill Ordered Observations

Here a missing sensor reading means no new reading arrived. Use the previous observation only if that assumption fits the task. Sort timestamps before filling.

In [ ]:
readings = pd.Series(
    [None, 20.0, None, None, 23.0],
    index=pd.date_range("2026-01-01", periods=5, freq="D"),
    name="temperature",
)
print(readings.ffill().to_string())

**Expected output**

```text
2026-01-01     NaN
2026-01-02    20.0
2026-01-03    20.0
2026-01-04    20.0
2026-01-05    23.0
Freq: D
```

The first value stays missing because no earlier observation exists. Do not carry values across unrelated devices or customers; fill within each group when appropriate.

## 7. Backward Fill and Its Limitation

Backward fill uses the next observed value. It can be useful for a retrospective task but can introduce future information into a prediction.

In [ ]:
readings = pd.Series([None, 20.0, None, 23.0, None])
print(readings.bfill().tolist())

**Expected output**

```text
[20.0, 20.0, 23.0, 23.0, nan]
```

The final value stays missing. Avoid backward filling when future observations would not have been available at prediction time.

## 8. Limit Replacements

`fillna(value, limit=...)` limits the total replacements in a Series. `ffill(limit=...)` limits each consecutive missing run.

In [ ]:
values = pd.Series([1.0, None, None, 4.0, None])
print("Constant fill, limit 1:", values.fillna(0, limit=1).tolist())
print("Forward fill, limit 1:", values.ffill(limit=1).tolist())

**Expected output**

```text
Constant fill, limit 1: [1.0, 0.0, nan, 4.0, nan]
Forward fill, limit 1: [1.0, 1.0, nan, 4.0, 4.0]
```

The two limits have different meanings. For a DataFrame, constant-fill limits apply along the selected axis.

## 9. Fill by Matching Index Labels

A replacement Series aligns by index label, not by its physical order.

In [ ]:
prices = pd.Series([None, 15.0, None], index=["A", "B", "C"])
defaults = pd.Series([30.0, 10.0], index=["C", "A"])
print(prices.fillna(defaults).to_string())

**Expected output**

```text
A    10.0
B    15.0
C    30.0
```

Use this when reliable replacement values are available for particular records.

## 10. Preserve the Original and Understand inplace

Assignment makes the result explicit. With `inplace=True`, the object is modified and the method returns `None`.

In [ ]:
original = pd.DataFrame({"discount": [5.0, None, 0.0]})
cleaned = original.fillna({"discount": 0})
print("Original missing count:", int(original["discount"].isna().sum()))
print("Cleaned missing count:", int(cleaned["discount"].isna().sum()))

working = original.copy()
returned = working.fillna({"discount": 0}, inplace=True)
print("Return value with inplace=True:", returned)
print("Working values:", working["discount"].tolist())

**Expected output**

```text
Original missing count: 1
Cleaned missing count: 0
Return value with inplace=True:    discount
0       5.0
1       0.0
2       0.0
Working values: [5.0, 0.0, 0.0]
```

Do not write `df = df.fillna(..., inplace=True)`; it assigns `None` to `df`. For a single column, prefer `df["column"] = df["column"].fillna(value)` over chained inplace operations.

## 11. Practical Example: Load a Sample Sales CSV

This synthetic CSV contains five orders. It is downloaded automatically from Plus2Net in the next code cell. No manual upload is required.

[Download the sample sales CSV](https://www.plus2net.com/python/download/pandas_fillna_sample_sales.csv)

The file is downloaded once per run of the loading cell. Later examples reload that local file to start with fresh data. If the download fails, check your connection and rerun the cell; if the file cannot be reached, do not continue with the sales examples.

Rules for this exercise:

- A blank category means "Unknown".
- A blank discount means zero discount; the discount is a total amount per order.
- A blank quantity or price remains unknown. Do not invent an order value.
- Keep all records and flag orders whose net amount cannot be calculated.

In real data, confirm these rules with the data owner first.

In [ ]:
from pathlib import Path
from urllib.request import urlopen

DATA_URL = "https://www.plus2net.com/python/download/pandas_fillna_sample_sales.csv"
DATA_FILE = Path("pandas_fillna_sample_sales.csv")

# Download first, then save only after a successful response.
with urlopen(DATA_URL, timeout=30) as response:
    csv_bytes = response.read()
DATA_FILE.write_bytes(csv_bytes)

def load_sales():
    data = pd.read_csv(DATA_FILE, dtype={"category": "string"})
    expected_columns = ["order_id", "category", "quantity", "unit_price", "discount"]
    if data.columns.tolist() != expected_columns:
        raise ValueError("The CSV columns do not match this lesson's sample dataset.")
    if len(data) != 5:
        raise ValueError("This lesson expects the five-order sample CSV.")
    return data

sales = load_sales()
print(sales.to_string(index=False))

**Expected output**

```text
 order_id   category  quantity  unit_price  discount
     1001 Stationery       2.0        20.0       0.0
     1002       <NA>       1.0        15.0       NaN
     1003 Stationery       NaN        10.0       0.0
     1004     Office       3.0         NaN       5.0
     1005     Office       2.0        30.0       NaN
```

## 12. Clean, Calculate and Verify

Apply only the agreed rules. Use fresh data and verify the outcome.

In [ ]:
sales = load_sales()
cleaned_sales = sales.fillna({"category": "Unknown", "discount": 0})
cleaned_sales["net_amount"] = (
    cleaned_sales["quantity"] * cleaned_sales["unit_price"]
    - cleaned_sales["discount"]
)
cleaned_sales["needs_review"] = cleaned_sales["net_amount"].isna()
print(cleaned_sales.to_string(index=False))
print("\nTotal for calculable orders:", cleaned_sales["net_amount"].sum(min_count=1))
print("Orders needing review:", int(cleaned_sales["needs_review"].sum()))

assert len(cleaned_sales) == len(sales)
assert cleaned_sales["order_id"].equals(sales["order_id"])
assert cleaned_sales["category"].isna().sum() == 0
assert cleaned_sales["discount"].isna().sum() == 0
assert cleaned_sales["quantity"].isna().sum() == 1
assert cleaned_sales["unit_price"].isna().sum() == 1
assert cleaned_sales["needs_review"].sum() == 2
assert pd.api.types.is_numeric_dtype(cleaned_sales["quantity"])
assert pd.api.types.is_numeric_dtype(cleaned_sales["unit_price"])
print("Verification passed.")

**Expected output**

```text
 order_id   category  quantity  unit_price  discount  net_amount  needs_review
     1001 Stationery       2.0        20.0       0.0        40.0         False
     1002    Unknown       1.0        15.0       0.0        15.0         False
     1003 Stationery       NaN        10.0       0.0         NaN          True
     1004     Office       3.0         NaN       5.0         NaN          True
     1005     Office       2.0        30.0       0.0        60.0         False

Total for calculable orders: 115.0
Orders needing review: 2
Verification passed.
```

The total is for calculable orders only; it is not the complete sales total. Keeping the two incomplete orders visible avoids silently hiding missing information.

### Download the Sample CSV and Your Results

The original CSV is already saved in the notebook runtime. Run this cell to save the cleaned results as a second CSV. In Colab, use the Files sidebar to download them. Runtime files are temporary, so download anything you want to keep.

In [ ]:
cleaned_sales.to_csv("pandas_fillna_cleaned_sales.csv", index=False)
print("Original sample: pandas_fillna_sample_sales.csv")
print("Saved results: pandas_fillna_cleaned_sales.csv")

## 13. Practice Questions

1. Count the missing values in each column of `load_sales()`.
2. Fill missing categories with "Uncategorised" and missing discounts with zero.
3. Leave missing quantities and prices unchanged. Explain why.
4. Add a Boolean `needs_review` column for orders with unknown quantity or price.
5. Count orders needing review and confirm no records were removed.
6. Compare the mean of `[10.0, None, 30.0]` before filling, after zero filling and after median filling.
7. Explain why forward fill leaves a leading missing value unchanged.
8. Explain when backward filling could leak future information.

### Practice Area

Try questions 1–5 before reading the suggested solution.

In [ ]:
# Practice area
practice = load_sales()
# Write your cleaning and verification steps here.


## 14. Suggested Solution: Sales Practice

This is one solution to questions 1–5. It applies explicit rules and retains incomplete orders.

In [ ]:
practice = load_sales()
print("Missing values before cleaning:")
print(practice.isna().sum().to_string())
answer = practice.fillna({"category": "Uncategorised", "discount": 0})
answer["needs_review"] = answer[["quantity", "unit_price"]].isna().any(axis=1)
print("\nOrders needing review:", int(answer["needs_review"].sum()))
print("Records retained:", len(answer))
assert len(answer) == len(practice)
assert answer["needs_review"].sum() == 2

**Expected output**

```text
Missing values before cleaning:
order_id      0
category      1
quantity      1
unit_price    1
discount      2

Orders needing review: 2
Records retained: 5
```

Other answers are possible if their assumptions are explained and the data checks pass.

## Key Takeaways

- Count missing values before changing the data.
- Choose replacements according to the meaning of each column.
- A missing value is not necessarily zero.
- Use `ffill()` and `bfill()` for appropriate ordered-data tasks.
- Median filling estimates a value; it does not recover the actual observation.
- Preserve original data and verify row counts, missing counts and numeric types.
- `inplace=True` returns `None`.
- Some values should remain missing until reliable information is available.

---

## Continue Learning

**Full Pandas fillna tutorial:**  
https://www.plus2net.com/python/pandas-dataframe-fillna.php

**Remove missing records with dropna:**  
https://www.plus2net.com/python/pandas-dataframe-dropna.php

**Understand summary statistics with describe:**  
https://www.plus2net.com/python/pandas-dataframe-describe.php

**More Python tutorials:**  
https://www.plus2net.com/python/site_map.php

**Official Pandas references:**  
https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.fillna.html  
https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.ffill.html  
https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.bfill.html

---

### plus2net Python Learning Series

This notebook accompanies the Plus2Net Pandas tutorial series.

© 2026 plus2net.com

**Validation note:** The revised notebook was executed locally with Pandas 3.0.1, using the supplied five-order CSV to simulate the download response. The loading logic, calculations and assertions passed. The hosted CSV could not be retrieved by the review tool, and live downloading and execution in Google Colab remain unverified. Expected outputs assume the hosted file matches the supplied sample. Learner-edited examples may produce different results.